# 零相位 USAF：双波长共享振幅

仅恢复一张共同振幅图；物体相位固定为零，探针和光谱权重已知。默认比较像素与 U-Net。
旧 OPD／独立吸收方向已备份到 backups，当前以 README.md 为准。


In [ ]:
from pathlib import Path
from dataclasses import replace
import sys

ROOT = Path.cwd()
if ROOT.name == 'multiwave':
    ROOT = ROOT.parent
if not (ROOT / 'multiwave' / 'config.py').exists():
    raise RuntimeError('请在 INNM_Code 或 INNM_Code/multiwave 目录启动 notebook')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from multiwave.config import Config, METHODS
from multiwave.run_simulation import run_experiment
from IPython.display import display, Image, Markdown


In [ ]:
cfg = Config.preset("smoke")
# 较大尺寸带噪声实验用 Config.preset("standard")
cfg = replace(cfg, iterations=80, device="auto")
# cfg = replace(cfg, wavelengths_nm=(515., 1030.), weights=(0.2, 0.8))
# cfg = replace(cfg, wavelengths_nm=(515.,), weights=(1.,))  # 单波长对照
methods = METHODS  # ("pixel_shared_amp", "unet_shared_amp")
cfg


In [ ]:
# 默认自动创建唯一时间戳目录，保存真值、重建、指标、模型和图像。
outdir, summary = run_experiment(cfg, methods)
display(Markdown((outdir / 'run_report.md').read_text(encoding='utf-8')))


In [ ]:
display(Image(filename=str(outdir / 'comparison.png')))
for method in methods:
    display(Image(filename=str(outdir / f'{method}_reconstruction.png')))


In [ ]:
# 一个共同物体：没有独立光谱吸收图，也没有物体相位优化。
for method in methods:
    metrics = summary[method]["final"]
    print(method)
    print("振幅相对误差:", metrics["shared_amplitude_relative_error"])
    print("振幅 PSNR:", metrics["shared_amplitude_psnr_db"])
    print("物体虚部最大值:", metrics["max_object_imaginary_abs"])
    print("波长间物体差异:", metrics["max_interwavelength_object_difference"])
